In [ ]:
#| default_exp core.precision

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import warnings
from dataclasses import asdict, dataclass

## Overview

Compression asks four questions. *What* block of parameters to remove is [granularity](granularity.html),
*which* ones matter is [criteria](criteria.html), *when* to act is [schedules](schedules.html) — and *how
few bits* to keep the survivors in is **precision**.

A precision request has five parts:

| Part | Argument | Values |
|---|---|---|
| Weight width | `weight_bits` | `4`, `8`, `16`, or a `{layer_name: width}` dict |
| Activation width | `act_bits` | `8`, `16` |
| Weight axis | `qscheme` | `'per_tensor'`, `'per_channel'`, `'per_group'` |
| Symmetry | `symmetric` | `True` (every zero-point is 0), `False` (affine) |
| Q/DQ placement | `qdq_placement` | `'per_op'`, `'skip_conv_add'` (the `pt2e` cell only) |

Two conventions run through the grammar:

- **A width of 16 means *not quantized***: the tensor keeps the model's floating-point dtype. One label
  then covers fp16 and fp32 models, so INT8 weight-only quantization is written `W8A16` whatever the
  model's float type. `weight_bits={'head': 16}` is how you keep one layer out of the quantization.
- **The finer the axis, the more scales**: `per_tensor` keeps one scale for the whole tensor,
  `per_channel` one per output channel, `per_group` one per fixed-size block of weights.
- **A placement is about *where the pairs sit*, not how many bits**: `per_op` quantizes the result of
  every operator the flow annotates; `skip_conv_add` leaves one edge — the residual branch's
  convolution feeding its add — without a Q/DQ pair, so that result reaches the add at accumulator
  precision. It changes the arithmetic, so it is opt-in and recorded on the spec.

Not every combination exists. A backend can only apply the cells its observers and kernels implement, and
only some of those survive an ONNX export. `PRECISION_SUPPORT` is that matrix, and `Quantizer` validates
every request against it **at construction**: a cell a backend cannot honor raises immediately, naming a
backend that can, instead of being silently rounded to something the backend *can* do.

## Precision cells

A `PrecisionCell` is one row of the matrix: a backend, a precision, and what that pair can do.

In [ ]:
#| export
QSCHEMES = ('per_tensor', 'per_channel', 'per_group')  # the weight axes this grammar names
WIDTHS = (4, 8, 16)                                    # the bit widths it names; 16 = left in floating point
QDQ_PLACEMENTS = ('per_op', 'skip_conv_add')  # where a flow may put its Q/DQ pairs


def _label(weight_bits: int, act_bits: int) -> str:
    "Short name of a precision, e.g. 'W8A8'"
    return f"W{weight_bits}A{act_bits}"


@dataclass(frozen=True, slots=True)
class PrecisionCell:
    "What one (backend, weight width, activation width) combination can do"
    backend: str
    weight_bits: int
    act_bits: int
    qschemes: tuple[str, ...]  # weight axes the cell can honor; the first one is its default
    symmetries: tuple[bool, ...]  # symmetry settings it can honor; the first one is its default
    per_layer: bool            # honors a {layer_name: width} dict
    exports: bool              # `export_qdq` can write it as a QDQ ONNX graph
    note: str                  # what it does, and why it does not export when it does not
    default_group_size: int | None = None  # group size the backend picks when the axis is 'per_group'
    qdq_placements: tuple[str, ...] = ()  # Q/DQ placements it can honor, first one its default

    @property
    def label(self) -> str:
        "Short name of the precision, e.g. 'W8A8'"
        return _label(self.weight_bits, self.act_bits)

    @property
    def default_qscheme(self) -> str:
        "Weight axis used when the caller does not name one"
        return self.qschemes[0]

    @property
    def default_symmetric(self) -> bool:
        "Symmetry used when the caller does not name one"
        return self.symmetries[0]

    @property
    def default_qdq_placement(self) -> str | None:
        "Q/DQ placement used when the caller does not name one, or None when the cell has no such axis"
        return self.qdq_placements[0] if self.qdq_placements else None

    def as_dict(self) -> dict:
        "Plain-dict view, for logging or serialization"
        return asdict(self)


_AFFINE_FX = ("FX observers keep activations affine (unsigned, non-zero zero-point), so the graph is not "
              "a portable Q/DQ export.")

_CELLS = (
    PrecisionCell('pt2e', 8, 8, ('per_channel', 'per_tensor'), (True, False), False, True,
                  "Symmetric INT8 weights and activations: every zero-point is 0, which is what `export_qdq` "
                  "writes into a QDQ ONNX graph. The one cell that also chooses where its Q/DQ pairs sit.",
                  qdq_placements=('per_op', 'skip_conv_add')),
    PrecisionCell('x86', 8, 8, ('per_channel', 'per_tensor'), (False,), True, False, _AFFINE_FX),
    PrecisionCell('fbgemm', 8, 8, ('per_channel', 'per_tensor'), (False,), True, False, _AFFINE_FX),
    PrecisionCell('onednn', 8, 8, ('per_channel', 'per_tensor'), (False,), True, False, _AFFINE_FX),
    PrecisionCell('qnnpack', 8, 8, ('per_tensor',), (False,), True, False, _AFFINE_FX),
    PrecisionCell('torchao', 8, 8, ('per_channel',), (True,), False, False,
                  "INT8 weights with dynamically quantized activations: the activation scales are computed at "
                  "run time, so they cannot be written into a static Q/DQ graph."),
    PrecisionCell('torchao', 8, 16, ('per_channel', 'per_group'), (True,), True, False,
                  "INT8 weight-only: the activations stay in floating point, so there is no activation Q/DQ "
                  "pair to export. This is the one torchao cell that honors a per-layer `weight_bits` "
                  "dict, over the Linear layers torchao rewrites."),
    PrecisionCell('torchao', 4, 16, ('per_group',), (True,), False, False,
                  "INT4 weight-only, a size lever: only torchao ships the kernels, and ONNX opset 18 has no "
                  "INT4 Q/DQ pair.", default_group_size=128),
)

PRECISION_SUPPORT: dict[tuple[str, int, int], PrecisionCell] = {
    (c.backend, c.weight_bits, c.act_bits): c for c in _CELLS}

_BACKENDS = tuple(dict.fromkeys(c.backend for c in _CELLS))
_LEGACY_BACKENDS = ('x86', 'fbgemm', 'onednn', 'qnnpack')


def _backends_where(predicate) -> list[str]:
    "Backends with at least one cell satisfying `predicate` — how a refusal names who CAN honor a request"
    return sorted({c.backend for c in _CELLS if predicate(c)})


def precision_table() -> str:
    "Markdown view of `PRECISION_SUPPORT` — the matrix is the code, this is only its rendering"
    head = ("| Backend | Precision | Weight axis | Symmetric | Per-layer | Q/DQ placement | "
            "`export_qdq` | Notes |\n"
            "|---|---|---|---|---|---|---|---|\n")
    rows = [f"| `{c.backend}` | {c.label} | {', '.join(c.qschemes)} | "
            f"{', '.join(str(s) for s in c.symmetries)} | {'yes' if c.per_layer else 'no'} | "
            f"{', '.join(c.qdq_placements) or 'n/a'} | "
            f"{'yes' if c.exports else 'no'} | {c.note} |" for c in _CELLS]
    return head + "\n".join(rows)

In [ ]:
show_doc(PrecisionCell)

## The support matrix

The table below is **rendered from `PRECISION_SUPPORT`** rather than written by hand, so it cannot drift
away from what the code enforces.

In [ ]:
from IPython.display import Markdown
Markdown(precision_table())

In [ ]:
show_doc(precision_table)

## The resolved spec

`_resolve_spec` turns a request into exactly one `QuantSpec`, or raises. `Quantizer` attaches the result
to the model it quantizes, where `quant_spec(model)` reads it back — that is what `export_qdq` consults
to refuse a precision it cannot write.

In [ ]:
#| export
SPEC_ATTR = '_fasterai_quant_spec'  # attribute `Quantizer` leaves on the models it quantizes


@dataclass(frozen=True, slots=True)
class QuantSpec:
    "The single precision cell a `Quantizer` resolved to — attached to the model it quantizes"
    backend: str
    method: str                         # 'static', 'dynamic', 'qat' or a torchao recipe
    weight_bits: int
    act_bits: int
    qscheme: str
    symmetric: bool                     # True when every zero-point is 0
    group_size: int | None = None       # weights sharing one scale (qscheme='per_group')
    layer_bits: dict | None = None      # per-layer widths, when the caller asked for some
    qdq_placement: str | None = None    # where the Q/DQ pairs sit; None on a backend with no such axis

    @property
    def label(self) -> str:
        "Short name of the precision, e.g. 'W8A8'"
        return _label(self.weight_bits, self.act_bits)

    @property
    def cell(self) -> PrecisionCell:
        "Row of `PRECISION_SUPPORT` this spec was validated against"
        return PRECISION_SUPPORT[(self.backend, self.weight_bits, self.act_bits)]

    @property
    def exports(self) -> bool:
        "Whether `export_qdq` can write this precision as a QDQ ONNX graph"
        return self.cell.exports

    @property
    def note(self) -> str:
        "What the cell does, and why it does not export when it does not"
        return self.cell.note

    def as_dict(self) -> dict:
        "Plain-dict view, for logging or serialization"
        return asdict(self)


def quant_spec(
    model,  # Any model, quantized or not
) -> QuantSpec | None:
    "The precision `Quantizer` applied to `model`, or `None` when fasterai did not quantize it"
    return getattr(model, SPEC_ATTR, None)

In [ ]:
show_doc(QuantSpec)

In [ ]:
show_doc(quant_spec)

## Resolution

One resolver, one place where a request is accepted or refused. Every refusal names the argument at fault
and, when there is one, a backend that can honor it.

In [ ]:
#| export
_TORCHAO_CELL = {'int8_weight_only': (8, 16), 'int8_dynamic': (8, 8), 'int4_weight_only': (4, 16)}
_TORCHAO_RECIPES = {_label(w, a): method for method, (w, a) in _TORCHAO_CELL.items()}


def _type_error(name: str, expectation: str, value) -> TypeError:
    "One shape for every type complaint, so they all read the same"
    return TypeError(f"`{name}` must be {expectation}, got {value!r} ({type(value).__name__}).")


def _check_width(name: str, value) -> int:
    "Validate one bit width, naming the argument that carried it"
    if isinstance(value, bool) or not isinstance(value, int):
        raise _type_error(name, f"an int, one of {list(WIDTHS)}", value)
    if value not in WIDTHS:
        raise ValueError(f"`{name}={value}` is not a width this grammar names. Use one of {list(WIDTHS)} "
                         "(16 means 'left in floating point').")
    return value


def _split_weight_bits(weight_bits) -> tuple[int | None, dict | None]:
    "Split `weight_bits` into a uniform width and the per-layer widths it may carry"
    if weight_bits is None: return None, None
    if isinstance(weight_bits, dict):
        if not weight_bits:
            raise ValueError("`weight_bits={}` names no layer. Pass an int for a uniform width, or a "
                             "{layer_name: width} dict with at least one entry.")
        for name, bits in weight_bits.items():
            if not isinstance(name, str):
                raise _type_error('weight_bits keys', 'layer names (str)', name)
            if _check_width(f"weight_bits['{name}']", bits) not in (8, 16):
                raise ValueError(f"`weight_bits['{name}']={bits}`: a per-layer width is 8 (quantize this "
                                 "layer) or 16 (leave it in floating point).")
        return (8 if 8 in weight_bits.values() else None), dict(weight_bits)
    return _check_width('weight_bits', weight_bits), None


def _resolve_method(backend: str, method: str, weight_bits: int | None,
                    act_bits: int | None) -> tuple[str, int, int]:
    "Reconcile `method` with the requested widths, and let the widths pick a torchao recipe"
    if backend != 'torchao':
        return method, weight_bits if weight_bits is not None else 8, act_bits if act_bits is not None else 8
    if method in _TORCHAO_CELL:
        w, a = _TORCHAO_CELL[method]
        for name, asked, native in (('weight_bits', weight_bits, w), ('act_bits', act_bits, a)):
            if asked is not None and asked != native:
                raise ValueError(
                    f"backend='torchao' method='{method}' is a {_label(w, a)} recipe, which "
                    f"`{name}={asked}` contradicts. Drop `{name}`, or name the recipe that matches: "
                    f"{_TORCHAO_RECIPES}.")
        return method, w, a
    if weight_bits is None and act_bits is None:
        raise ValueError(f"backend='torchao' has no method '{method}'. Its recipes are "
                         f"{sorted(_TORCHAO_CELL)}, or name the precision directly with "
                         "`weight_bits=` / `act_bits=`.")
    w = weight_bits if weight_bits is not None else 8
    a = act_bits if act_bits is not None else (16 if w == 4 else 8)
    derived = _TORCHAO_RECIPES.get(_label(w, a))
    if derived is None:
        raise ValueError(f"backend='torchao' has no recipe for {_label(w, a)}. It ships "
                         f"{_TORCHAO_RECIPES}.")
    return derived, w, a


def _lookup_cell(backend: str, weight_bits: int, act_bits: int) -> PrecisionCell:
    "Row of `PRECISION_SUPPORT` for this precision, or the reason there is none"
    cell = PRECISION_SUPPORT.get((backend, weight_bits, act_bits))
    if cell is not None: return cell
    label = _label(weight_bits, act_bits)
    if backend not in _BACKENDS:
        raise ValueError(f"Unknown backend '{backend}'. fasterai quantizes with {list(_BACKENDS)}.")
    here = [c.label for c in _CELLS if c.backend == backend]
    elsewhere = _backends_where(lambda c: (c.weight_bits, c.act_bits) == (weight_bits, act_bits))
    if elsewhere:
        raise ValueError(f"backend='{backend}' cannot quantize {label}; it runs {here}. "
                         f"The backend(s) that run {label}: {elsewhere}.")
    raise ValueError(f"No fasterai backend runs {label}: nothing ships a kernel for it. "
                     f"backend='{backend}' runs {here}.")


def _resolve_qscheme(cell: PrecisionCell, qscheme, group_size, use_per_tensor: bool) -> str:
    "Pick the weight axis, refusing any the cell cannot honor"
    if qscheme is not None:
        if not isinstance(qscheme, str):
            raise _type_error('qscheme', f"one of {list(QSCHEMES)} (str)", qscheme)
        if qscheme not in QSCHEMES:
            raise ValueError(f"Unknown qscheme '{qscheme}'. The weight axes this grammar names are "
                             f"{list(QSCHEMES)}.")
    if use_per_tensor:
        if cell.backend not in _LEGACY_BACKENDS:
            raise ValueError(f"`use_per_tensor=True` is a legacy-backend flag that backend='{cell.backend}' "
                             "never read. Ask for the axis instead: qscheme='per_tensor'.")
        if qscheme not in (None, 'per_tensor'):
            raise ValueError(f"`use_per_tensor=True` and `qscheme='{qscheme}'` ask for different weight "
                             "axes. Keep one.")
        qscheme = 'per_tensor'
    if qscheme is None:
        qscheme = 'per_group' if (group_size is not None and 'per_group' in cell.qschemes) \
            else cell.default_qscheme
    if qscheme not in cell.qschemes:
        able = _backends_where(lambda c: qscheme in c.qschemes)
        raise ValueError(f"backend='{cell.backend}' {cell.label} quantizes weights {list(cell.qschemes)}, "
                         f"not '{qscheme}'. The backend(s) that do: {able}.")
    return qscheme


def _resolve_group_size(cell: PrecisionCell, qscheme: str, group_size) -> int | None:
    "Validate the group size against the axis that gives it a meaning"
    if group_size is None:
        if qscheme != 'per_group': return None
        if cell.default_group_size is None:
            raise ValueError("qscheme='per_group' needs a `group_size` (e.g. group_size=128): a group is a "
                             "fixed number of weights sharing one scale.")
        return cell.default_group_size
    if isinstance(group_size, bool) or not isinstance(group_size, int):
        raise _type_error('group_size', 'a positive int', group_size)
    if group_size <= 0:
        raise ValueError(f"`group_size={group_size}` is not a size: a group holds at least one weight.")
    if qscheme != 'per_group':
        able = _backends_where(lambda c: 'per_group' in c.qschemes)
        raise ValueError(f"`group_size={group_size}` only means something with qscheme='per_group'; "
                         f"backend='{cell.backend}' {cell.label} quantizes weights '{qscheme}'. "
                         f"The backend(s) that quantize per group: {able}.")
    return group_size


def _resolve_qdq_placement(cell: PrecisionCell, qdq_placement) -> str | None:
    "Pick where the Q/DQ pairs sit, refusing a placement the cell's flow cannot produce"
    if qdq_placement is None: return cell.default_qdq_placement
    if not isinstance(qdq_placement, str):
        raise _type_error('qdq_placement', f"one of {list(QDQ_PLACEMENTS)} (str)", qdq_placement)
    if qdq_placement not in QDQ_PLACEMENTS:
        raise ValueError(f"Unknown qdq_placement '{qdq_placement}'. The Q/DQ placements this grammar "
                         f"names are {list(QDQ_PLACEMENTS)}.")
    if qdq_placement not in cell.qdq_placements:
        able = _backends_where(lambda c: qdq_placement in c.qdq_placements)
        # first arm: a cell that names SOME placements but not this one — none is in that state today
        why = (f"it places them {list(cell.qdq_placements)}" if cell.qdq_placements else
               "its flow quantizes every operator it rewrites and names no placement at all")
        raise ValueError(f"backend='{cell.backend}' {cell.label} cannot honor "
                         f"qdq_placement='{qdq_placement}' — {why}. The backend(s) that can: {able}.")
    return qdq_placement


def _resolve_symmetry(cell: PrecisionCell, symmetric) -> bool:
    "Pick the symmetry, refusing the one the cell's observers cannot produce"
    if symmetric is None: return cell.default_symmetric
    if not isinstance(symmetric, bool):
        raise _type_error('symmetric', 'True, False or None', symmetric)
    if symmetric not in cell.symmetries:
        able = _backends_where(lambda c: symmetric in c.symmetries)
        why = ("its observers are affine by construction: activations carry a non-zero zero-point"
               if symmetric else "it quantizes symmetrically by construction")
        raise ValueError(f"backend='{cell.backend}' {cell.label} cannot honor symmetric={symmetric} — "
                         f"{why}. The backend(s) that can: {able}.")
    if not symmetric and cell.exports:
        warnings.warn("symmetric=False keeps the activations affine: the exported graph then carries "
                      "non-zero zero-points, which some runtimes refuse.", UserWarning, stacklevel=2)
    return symmetric


def _resolve_spec(
    backend: str = 'x86',
    method: str = 'static',
    *,
    weight_bits=None,
    act_bits=None,
    qscheme: str | None = None,
    group_size: int | None = None,
    symmetric: bool | None = None,
    qdq_placement: str | None = None,
    use_per_tensor: bool = False,
) -> QuantSpec:
    "Resolve the precision grammar into the one `QuantSpec` a backend will apply, or say why it cannot"
    if not isinstance(backend, str):
        raise _type_error('backend', 'a str', backend)
    weight_bits, layer_bits = _split_weight_bits(weight_bits)
    if act_bits is not None: _check_width('act_bits', act_bits)
    method, weight_bits, act_bits = _resolve_method(backend, method, weight_bits, act_bits)
    cell = _lookup_cell(backend, weight_bits, act_bits)
    qscheme = _resolve_qscheme(cell, qscheme, group_size, use_per_tensor)
    group_size = _resolve_group_size(cell, qscheme, group_size)
    symmetric = _resolve_symmetry(cell, symmetric)
    qdq_placement = _resolve_qdq_placement(cell, qdq_placement)
    if layer_bits and not cell.per_layer:
        sibling = next((c for c in _CELLS if c.backend == backend and c.per_layer), None)
        if sibling is not None:
            raise ValueError(f"backend='{backend}' cannot honor a per-layer `weight_bits` dict at "
                             f"{cell.label}, only at {sibling.label}: add act_bits={sibling.act_bits}.")
        raise ValueError(f"backend='{backend}' quantizes the whole model at once: it cannot honor a "
                         f"per-layer `weight_bits` dict. The backend(s) that can: "
                         f"{_backends_where(lambda c: c.per_layer)}.")
    return QuantSpec(backend=backend, method=method, weight_bits=weight_bits, act_bits=act_bits,
                     qscheme=qscheme, symmetric=symmetric, group_size=group_size, layer_bits=layer_bits,
                     qdq_placement=qdq_placement)

---

## Usage Examples

The grammar is reached through `Quantizer`, which forwards its precision arguments to `_resolve_spec`:

```python
from fasterai.quantize.quantizer import Quantizer
from fasterai.core.all import quant_spec

# The deployable cell: symmetric INT8, per-channel weights, exportable as QDQ ONNX
quantizer = Quantizer(backend='pt2e', weight_bits=8, act_bits=8, qscheme='per_channel', symmetric=True)
model_q = quantizer.quantize(model, calibration_dl=dls.valid)
print(quant_spec(model_q).as_dict())
# {'backend': 'pt2e', 'method': 'static', 'weight_bits': 8, 'act_bits': 8, 'qscheme': 'per_channel',
#  'symmetric': True, 'group_size': None, 'layer_bits': None, 'qdq_placement': 'per_op'}

# INT8 weight-only, one scale per group of 64 weights
Quantizer(backend='torchao', weight_bits=8, act_bits=16, qscheme='per_group', group_size=64)

# Keep the classifier out of the quantization (16 = left in floating point)
Quantizer(backend='x86', weight_bits={'fc': 16})

# The same dict on torchao, over the Linear layers it rewrites: INT8 everywhere except one layer
Quantizer(backend='torchao', weight_bits={'layers.0.linear1': 16}, act_bits=16)
```

A `{layer_name: width}` dict says *which* layers are quantized; the width they are quantized *at* is the
uniform one, so the two spellings below resolve to the same cell. Layers the dict does not name keep that
uniform width — the dict is an override list, not a whitelist.

```python
_resolve_spec('torchao', weight_bits={'fc': 16}, act_bits=16).weight_bits          # 8
_resolve_spec('torchao', weight_bits={'fc': 16, 'head': 8}, act_bits=16).weight_bits  # 8
```

`qdq_placement` names where the Q/DQ pairs sit rather than how many bits they keep. Only the `pt2e`
cell has that axis, so it is the only one whose spec carries a value; everywhere else the field stays
`None` and naming a placement raises:

```python
Quantizer(backend='pt2e', qdq_placement='skip_conv_add')  # the residual conv->add edge stays unquantized
_resolve_spec('pt2e').qdq_placement                       # 'per_op' — asking for it explicitly is the same request
_resolve_spec('x86').qdq_placement                        # None — the FX flow has no such axis
```

Asking for a cell a backend cannot honor raises, and says who can:

```python
Quantizer(backend='x86', symmetric=True)
# ValueError: backend='x86' W8A8 cannot honor symmetric=True — its observers are affine by
# construction: activations carry a non-zero zero-point. The backend(s) that can: ['pt2e', 'torchao'].

Quantizer(backend='torchao', weight_bits={'fc': 8})   # W8A8: torchao's dynamic-activation recipe
# ValueError: backend='torchao' cannot honor a per-layer `weight_bits` dict at W8A8, only at W8A16:
# add act_bits=16.
```

---

## See Also

- [Granularity](granularity.html) - What block of parameters to remove
- [Criteria](criteria.html) - Which parameters matter
- [Schedules](schedules.html) - When compression happens
- [Quantizer](../quantize/quantizer.html) - The class that applies a precision cell
- [ONNX Exporter](../export/onnx_exporter.html) - `export_qdq`, which refuses the cells it cannot write

Tests live in `nbs/tests/test_precision.ipynb`.